### Purpose

The purpose of this notebook is to quantify spatial relationships between oil and gas wells and saltwater injection wells (SWIW) across the study area.

### Load Libraries

In [58]:
import pandas as pd
import numpy as np
from geopy.distance import geodesic
import geopandas as gpd
from shapely.geometry import Point
from tqdm import tqdm

### Load Data

In [59]:
well_info = pd.read_csv('../data/final/well_information_final.csv')
permits = pd.read_csv('../data/final/permit_plug_list_final.csv')

### Create GeoDataFrames for each dataset

In [81]:
well_info = well_info.dropna(subset=['WELL LATITUDE', 'WELL LONGITUDE'])
permits = permits.dropna(subset=['WH_LAT', 'WH_LONG'])

well_gdf = gpd.GeoDataFrame(
    well_info,
    geometry=gpd.points_from_xy(
        well_info['WELL LONGITUDE'], well_info['WELL LATITUDE']
    ),
    crs='EPSG:4326'
)

permit_gdf = gpd.GeoDataFrame(
    permits,
    geometry=gpd.points_from_xy(
        permits['WH_LONG'], permits['WH_LAT']
    ),
    crs='EPSG:4326'
)


### Identify (SWIW)

In [82]:
saltwater_gdf = permit_gdf[
    permit_gdf['PROP_WLTYPE'] == 'SW_R'
]

well_gdf = well_gdf[~well_gdf['API WELL NUMBER'].isin(saltwater_gdf['API_WELLNO'])]

print(f"Total wells: {len(well_gdf)}")
print(f"Saltwater wells: {len(saltwater_gdf)}")

Total wells: 73595
Saltwater wells: 397


### Compute pairwise distances between wells

In [83]:
def nearest_distance(source_gdf, target_gdf, src_col, target_col):
    target_sindex = target_gdf.sindex
    nearest_list = []

    for idx, src_row in tqdm(source_gdf.iterrows(), total=len(source_gdf), desc=f'Finding nearest {target_col}'):
        nearest_idx = list(target_sindex.nearest(src_row.geometry, 1))
        nearest_geom = target_gdf.iloc[nearest_idx[0]].geometry

        distance_km = src_row.geometry.distance(nearest_geom) * 111

        nearest_list.append({
            src_col: src_row[src_col],
            f'Nearest_{target_col}_km': distance_km.iloc[0]
        })

    return pd.DataFrame(nearest_list)

### Compute Nearest Wells and Export

In [84]:
nearest_swiw_df = nearest_distance(well_gdf, saltwater_gdf, 'API WELL NUMBER', 'SWIW')
nearest_swiw_df.to_csv('../data/distance/well_to_nearest_saltwater.csv', index=False)

nearest_well_df = nearest_distance(well_gdf, well_gdf, 'API WELL NUMBER', 'Well')
nearest_well_df.to_csv('../data/distance/well_to_nearest_well.csv', index=False)

Finding nearest Well: 100%|██████████| 73595/73595 [00:48<00:00, 1517.65it/s]


In [85]:
nearest_swiw_df.head(5)

,API WELL NUMBER,Nearest_SWIW_km
0,34001900060000,200.469292
1,34003200520000,19.408226
2,34003641500000,24.655112
3,34003200460000,44.114159
4,34003200670000,32.231274


In [93]:
# Basic Regression - Distance to Nearest Saltwater Injection Well vs. Well Production

import statsmodels.api as sm
distance_data = pd.read_csv('../data/distance/well_to_nearest_saltwater.csv')
production_data = pd.read_csv('../data/final/annual_production_final.csv')
regression_data = pd.merge(distance_data, production_data, left_on='API WELL NUMBER', right_on='API Number')

regression_data = regression_data[(regression_data['Production Year'] == 2024) & (regression_data['Quarter (1, 2, 3, 4, n/a)'].isna())]

X = regression_data['Nearest_SWIW_km']
y = regression_data['Gas (Mcf)']
X = sm.add_constant(X)
model = sm.OLS(y, X).fit()
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:              Gas (Mcf)   R-squared:                       0.019
Model:                            OLS   Adj. R-squared:                  0.019
Method:                 Least Squares   F-statistic:                     684.5
Date:                Sat, 08 Nov 2025   Prob (F-statistic):          1.85e-149
Time:                        23:17:11   Log-Likelihood:            -3.1495e+05
No. Observations:               35368   AIC:                         6.299e+05
Df Residuals:                   35366   BIC:                         6.299e+05
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const            -687.9218     52.697    -

In [91]:
# Same thing for oil production
distance_data = pd.read_csv('../data/distance/well_to_nearest_saltwater.csv')
production_data = pd.read_csv('../data/final/annual_production_final.csv')
regression_data = pd.merge(distance_data, production_data, left_on='API WELL NUMBER', right_on='API Number')

regression_data = regression_data[(regression_data['Production Year'] == 2024) & (regression_data['Quarter (1, 2, 3, 4, n/a)'].isna()) & (regression_data['Oil (Bbls)'] > 0)]

X = regression_data['Nearest_SWIW_km']
y = regression_data['Oil (Bbls)']
X = sm.add_constant(X)
model = sm.OLS(y, X).fit()
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:             Oil (Bbls)   R-squared:                       0.011
Model:                            OLS   Adj. R-squared:                  0.011
Method:                 Least Squares   F-statistic:                     152.7
Date:                Sat, 08 Nov 2025   Prob (F-statistic):           6.94e-35
Time:                        23:16:14   Log-Likelihood:                -92908.
No. Observations:               13594   AIC:                         1.858e+05
Df Residuals:                   13592   BIC:                         1.858e+05
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const             243.5872     10.162     